# Econometría Financiera: Series de Tiempo y Volatilidad

Versión reconstruida con un stack econométrico apropiado: `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`, `seaborn` y `sympy`.

**Audiencia**
- Estudiantes de economía, finanzas cuantitativas, estadística aplicada y ciencia de datos.

**Prerrequisitos**
- Probabilidad e inferencia.
- Series de tiempo básicas.
- Regresión lineal y álgebra matricial.

**Objetivos**
- Simular y diagnosticar procesos con media y varianza condicional.
- Aplicar pruebas de estacionariedad, ACF/PACF, Ljung-Box y ARCH-LM.
- Estimar la media con `statsmodels` y la volatilidad con QMLE en `scipy`.
- Construir gráficos y animaciones que muestren clustering de volatilidad.
- Traducir el modelo en medidas de riesgo como VaR y Expected Shortfall.


## Tabla de Contenidos

1. Configuración y utilidades.
2. Hechos estilizados de rendimientos.
3. Dinámica de la media: ARIMA, ADF, ACF y PACF.
4. Derivación simbólica de la varianza incondicional del GARCH.
5. Estimación cuasi-máxima verosímil con `scipy`.
6. VaR, ES y backtesting básico.
7. Animación del clustering de volatilidad.
8. Ejercicios.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")
os.environ.setdefault("MPLBACKEND", "Agg")

import numpy as np
import pandas as pd
import scipy.stats as st
import sympy as sp
import seaborn as sns
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from scipy.optimize import minimize
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller

sns.set_theme(style="whitegrid", context="talk")
rng = np.random.default_rng(20260316)


def display_animation(anim):
    html = HTML(anim.to_jshtml(default_mode="loop"))
    plt.close(anim._fig)
    display(html)


def simulate_arma_garch(
    n=850,
    burn=200,
    mu=0.0003,
    phi=0.18,
    theta=-0.10,
    omega=0.000006,
    alpha=0.08,
    beta=0.90,
    df=7,
    seed=20260316,
):
    local_rng = np.random.default_rng(seed)
    total = n + burn
    z = st.t.rvs(df=df, size=total, random_state=local_rng)
    z = z / np.sqrt(df / (df - 2))
    e = np.zeros(total)
    h = np.zeros(total)
    r = np.zeros(total)
    h[0] = omega / (1 - alpha - beta)
    for t in range(1, total):
        h[t] = omega + alpha * e[t - 1] ** 2 + beta * h[t - 1]
        e[t] = np.sqrt(h[t]) * z[t]
        r[t] = mu + phi * (r[t - 1] - mu) + theta * e[t - 1] + e[t]
    ret = r[burn:]
    var = h[burn:]
    price = 100 * np.exp(np.cumsum(ret))
    df_out = pd.DataFrame(
        {
            "return": ret,
            "variance_true": var,
            "price": price,
        }
    )
    df_out["rolling_vol_30"] = df_out["return"].rolling(30).std()
    df_out["squared_return"] = df_out["return"] ** 2
    return df_out


def garch_variance(params, resid):
    omega, alpha, beta = params
    if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 0.999:
        return None
    h = np.empty_like(resid)
    h[0] = np.var(resid)
    for t in range(1, len(resid)):
        h[t] = omega + alpha * resid[t - 1] ** 2 + beta * h[t - 1]
    return np.clip(h, 1e-10, None)


def garch_negloglike(params, resid):
    h = garch_variance(params, resid)
    if h is None:
        return 1e12
    return 0.5 * np.sum(np.log(h) + resid**2 / h)


def fit_garch(resid):
    resid = np.asarray(resid, dtype=float)
    x0 = np.array([1e-5, 0.08, 0.90])
    bounds = [(1e-8, 1e-2), (1e-6, 0.4), (1e-6, 0.995)]
    cons = {"type": "ineq", "fun": lambda x: 0.999 - x[1] - x[2]}
    result = minimize(
        garch_negloglike,
        x0=x0,
        args=(resid,),
        method="SLSQP",
        bounds=bounds,
        constraints=[cons],
        options={"maxiter": 300, "ftol": 1e-9},
    )
    params = result.x
    h = garch_variance(params, resid)
    return {"result": result, "params": params, "variance": h}


print("Stack econométrico listo.")


## 1. Hechos estilizados: rendimientos, colas y clustering

Los rendimientos financieros suelen presentar:

- media pequeña;
- baja autocorrelación lineal en retornos;
- alta autocorrelación en $|r_t|$ y $r_t^2$;
- colas más pesadas que la normal;
- episodios de volatilidad persistente.

En vez de usar una serie artificial mínima, generamos una trayectoria sintética con media ARMA y varianza GARCH. Esto permite diagnosticar con herramientas reales y obtener gráficas informativas.


In [ ]:
finance_df = simulate_arma_garch()

fig, axes = plt.subplots(2, 2, figsize=(16, 10), facecolor="#fbfaf7")
sns.lineplot(data=finance_df, x=finance_df.index, y="price", ax=axes[0, 0], color="#1f77b4")
axes[0, 0].set_title("Precio sintético")
axes[0, 0].set_xlabel("t")
axes[0, 0].set_ylabel("Precio")

sns.lineplot(data=finance_df, x=finance_df.index, y="return", ax=axes[0, 1], color="#c44e52")
axes[0, 1].axhline(0, color="black", lw=1, alpha=0.6)
axes[0, 1].set_title("Rendimientos")
axes[0, 1].set_xlabel("t")
axes[0, 1].set_ylabel("r_t")

sns.histplot(finance_df["return"], kde=True, bins=40, ax=axes[1, 0], color="#4c72b0")
axes[1, 0].set_title("Distribución de rendimientos")

sns.lineplot(data=finance_df, x=finance_df.index, y="rolling_vol_30", ax=axes[1, 1], color="#dd8452")
axes[1, 1].set_title("Volatilidad móvil (30)")
axes[1, 1].set_xlabel("t")
axes[1, 1].set_ylabel("desvío estándar")

fig.tight_layout()
plt.show()

print(finance_df[["price", "return", "rolling_vol_30"]].describe().round(4))


## 2. Media condicional: ADF, ACF/PACF y ARIMA

En la práctica, la modelación financiera suele separar:

- una ecuación para la media condicional;
- una ecuación para la varianza condicional.

Para la media usaremos `statsmodels`:
- prueba ADF para estacionariedad;
- ACF/PACF para estructura serial;
- un ARIMA simple para capturar la dependencia lineal.


In [ ]:
adf_stat, adf_pvalue, _, _, crit_values, _ = adfuller(finance_df["return"])

arima_res = ARIMA(finance_df["return"], order=(1, 0, 1)).fit()
lb_df = acorr_ljungbox(arima_res.resid, lags=[10], return_df=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), facecolor="#fbfaf7")
plot_acf(finance_df["return"], lags=24, ax=axes[0])
plot_pacf(finance_df["return"], lags=24, ax=axes[1], method="ywm")
axes[0].set_title("ACF de los rendimientos")
axes[1].set_title("PACF de los rendimientos")
fig.tight_layout()
plt.show()

arima_table = pd.DataFrame(
    {
        "coef": arima_res.params,
        "std_err": arima_res.bse,
        "p_value": arima_res.pvalues,
    }
).round(5)

print("ADF statistic:", round(adf_stat, 4))
print("ADF p-value:", round(adf_pvalue, 6))
print("Valores críticos:", {k: round(v, 4) for k, v in crit_values.items()})
print()
print("Coeficientes ARIMA(1,0,1):")
print(arima_table)
print()
print("Ljung-Box Q(10) sobre residuos ARIMA:")
print(lb_df.round(5))


## 3. Derivación simbólica: la varianza incondicional del GARCH(1,1)

Si
\[
h_t = \omega + \alpha \varepsilon_{t-1}^2 + \beta h_{t-1},
\]
y bajo estacionariedad $\mathbb{E}[\varepsilon_t^2] = \mathbb{E}[h_t] = \bar h$, entonces:
\[
\bar h = \omega + (\alpha + \beta)\bar h
\quad \Rightarrow \quad
\bar h = \frac{\omega}{1-\alpha-\beta}.
\]

Aquí usamos `sympy` no para “adornar”, sino para dejar la derivación explícita y reproducible.


In [ ]:
omega_s, alpha_s, beta_s, hbar = sp.symbols("omega alpha beta hbar", positive=True)
eq = sp.Eq(hbar, omega_s + (alpha_s + beta_s) * hbar)
solution = sp.solve(eq, hbar)[0]

display(eq)
display(sp.Eq(hbar, sp.simplify(solution)))


## 4. Estimación de volatilidad: QMLE con `scipy` y diagnóstico con `statsmodels`

Usaremos un esquema realista:

1. estimamos la media con ARIMA;
2. tomamos los residuos;
3. estimamos GARCH(1,1) por cuasi-máxima verosimilitud con `scipy.optimize.minimize`;
4. contrastamos heterocedasticidad condicional con `het_arch`.

Esto corrige precisamente la limitación de una notebook basada solo en funciones hechas “a mano”: ahora el flujo es similar al de un trabajo empírico serio.


In [ ]:
resid = arima_res.resid.to_numpy()
garch_fit = fit_garch(resid)
omega_hat, alpha_hat, beta_hat = garch_fit["params"]
h_hat = garch_fit["variance"]
standardized = resid / np.sqrt(h_hat)

arch_before = het_arch(resid, nlags=8)
arch_after = het_arch(standardized, nlags=8)

fig, axes = plt.subplots(2, 1, figsize=(14, 9), facecolor="#fbfaf7")
sns.lineplot(x=np.arange(len(resid)), y=resid**2, ax=axes[0], label="residuo^2", color="#c44e52")
sns.lineplot(x=np.arange(len(h_hat)), y=h_hat, ax=axes[0], label="varianza condicional estimada", color="#55a868")
axes[0].set_title("Residuos cuadrados vs. varianza GARCH estimada")
axes[0].set_xlabel("t")

st.probplot(standardized, dist="norm", plot=axes[1])
axes[1].set_title("QQ-plot de residuos estandarizados")
fig.tight_layout()
plt.show()

params_table = pd.DataFrame(
    {
        "estimacion": [omega_hat, alpha_hat, beta_hat, alpha_hat + beta_hat],
    },
    index=["omega", "alpha", "beta", "persistencia"],
).round(6)

print("Parámetros GARCH(1,1):")
print(params_table)
print()
print("ARCH-LM antes del ajuste:", {"LM": round(arch_before[0], 4), "p_value": round(arch_before[1], 6)})
print("ARCH-LM después del ajuste:", {"LM": round(arch_after[0], 4), "p_value": round(arch_after[1], 6)})


## 5. VaR, Expected Shortfall y backtesting simple

Con la media y la varianza condicional estimadas podemos construir:

- **Value-at-Risk** al 1%:
  \[
  VaR_{0.01,t} = \mu_t + z_{0.01}\sqrt{h_t}.
  \]
- **Expected Shortfall** normal:
  \[
  ES_{0.01,t} = \mu_t - \frac{\varphi(z_{0.01})}{0.01}\sqrt{h_t}.
  \]

Luego comparamos el VaR in-sample con los retornos observados para obtener una verificación descriptiva de excedencias.


In [ ]:
mu_fitted = np.asarray(arima_res.fittedvalues)
q01 = st.norm.ppf(0.01)
var_series = mu_fitted + q01 * np.sqrt(h_hat)
es_series = mu_fitted - st.norm.pdf(q01) / 0.01 * np.sqrt(h_hat)
exceedance_rate = (finance_df["return"].to_numpy() < var_series).mean()

fig, ax = plt.subplots(figsize=(14, 5), facecolor="#fbfaf7")
sns.lineplot(x=finance_df.index, y=finance_df["return"], ax=ax, color="#4c72b0", label="retorno")
sns.lineplot(x=finance_df.index, y=var_series, ax=ax, color="#c44e52", label="VaR 1%")
sns.lineplot(x=finance_df.index, y=es_series, ax=ax, color="#8172b3", label="ES 1%")
ax.fill_between(finance_df.index, var_series, es_series, color="#f1c0c0", alpha=0.25)
ax.set_title("Retornos vs. VaR y ES condicionales")
ax.set_xlabel("t")
ax.set_ylabel("retorno")
plt.show()

risk_table = pd.DataFrame(
    {
        "media_condicional": [mu_fitted[-1]],
        "var_1pct": [var_series[-1]],
        "es_1pct": [es_series[-1]],
        "tasa_excedencia": [exceedance_rate],
    }
).round(5)
print(risk_table.T)


## 6. Animación: cómo se acumulan los episodios de alta volatilidad

En vez de una figura estática, esta animación deja ver algo central en finanzas:
la varianza condicional no se mueve “por ruido”, sino que entra en regímenes de alta y baja persistencia. La gráfica superior acumula la trayectoria de retornos y la inferior la volatilidad móvil.


In [ ]:
frame_idx = np.linspace(40, len(finance_df) - 1, 90, dtype=int)
fig, axes = plt.subplots(2, 1, figsize=(12, 8), facecolor="#fbfaf7")

for ax in axes:
    ax.set_xlim(0, len(finance_df))

axes[0].set_ylim(finance_df["return"].min() * 1.2, finance_df["return"].max() * 1.2)
axes[1].set_ylim(0, finance_df["rolling_vol_30"].max() * 1.1)
axes[0].set_title("Retornos acumulados en el tiempo")
axes[1].set_title("Volatilidad móvil y clustering")

line_ret, = axes[0].plot([], [], color="#4c72b0", lw=2)
dot_ret, = axes[0].plot([], [], "o", color="#c44e52")
line_vol, = axes[1].plot([], [], color="#dd8452", lw=2)
dot_vol, = axes[1].plot([], [], "o", color="#55a868")

def update(frame_number):
    idx = frame_idx[frame_number]
    x_data = finance_df.index[: idx + 1]
    y_ret = finance_df["return"].iloc[: idx + 1]
    y_vol = finance_df["rolling_vol_30"].iloc[: idx + 1]

    line_ret.set_data(x_data, y_ret)
    dot_ret.set_data([idx], [finance_df["return"].iloc[idx]])
    line_vol.set_data(x_data, y_vol)
    dot_vol.set_data([idx], [finance_df["rolling_vol_30"].iloc[idx]])
    return line_ret, dot_ret, line_vol, dot_vol

anim = FuncAnimation(fig, update, frames=len(frame_idx), interval=90, blit=True)
display_animation(anim)


## 7. Ejercicios

1. Cambia los parámetros del GARCH a $(\alpha,\beta)=(0.05,0.70)$ y compara persistencia.
2. Estima un ARIMA alternativo y compara AIC/BIC.
3. Repite el cálculo de VaR con cuantiles t-Student en vez de normalidad.
4. Contrasta el ARCH-LM para residuos brutos, residuos ARIMA y residuos estandarizados.


In [ ]:
# Celda guía para experimentar.

alt_df = simulate_arma_garch(alpha=0.05, beta=0.70, seed=20260317)
alt_model = ARIMA(alt_df["return"], order=(1, 0, 1)).fit()
alt_fit = fit_garch(alt_model.resid.to_numpy())

print("Persistencia alternativa:", round(alt_fit["params"][1] + alt_fit["params"][2], 4))
print("AIC del ARIMA alternativo:", round(alt_model.aic, 3))
print("ARCH-LM residuos alternativos:", het_arch(alt_model.resid, nlags=8)[1])


## Referencias

- Tsay, R. S. *Analysis of Financial Time Series*.
- Hamilton, J. D. *Time Series Analysis*.
- Engle, R. F. (1982). *ARCH*.
- Bollerslev, T. (1986). *GARCH*.
